In [1]:
import os
import joblib
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = r"E:\Praxis\TERM2\DAS\retail_parquet"
MODELS_DIR = os.path.join(BASE_DIR, "mlops_pipeline", "models")
MODEL_PATH = os.path.join(MODELS_DIR, "late_delivery_model.pkl")

class DeliveryPredictor:
    def __init__(self):
        if not os.path.exists(MODEL_PATH):
            raise FileNotFoundError(f"Model not found at {MODEL_PATH}. Please run train.py first.")
        self.model = joblib.load(MODEL_PATH)
        print("Model loaded successfully.")

    def predict(self, order_data: dict):
        """
        Expects a dictionary with the following keys:
        'revenue', 'freight', 'items_count', 'product_weight_g', 'product_volume',
        'customer_state', 'seller_state', 'purchase_month', 'purchase_dayofweek'
        """
        df = pd.DataFrame([order_data])
        
        prob_late = self.model.predict_proba(df)[0][1]
        is_late = int(self.model.predict(df)[0])
        
        result = {
            "is_late_prediction": is_late,
            "probability_late": float(prob_late),
            "risk_level": "High" if prob_late > 0.6 else "Medium" if prob_late > 0.4 else "Low",
            "recommended_action": "Subsidize Expedited Shipping" if is_late == 1 else "Standard Routing"
        }
        return result

if __name__ == "__main__":
    # Simulate an incoming order
    dummy_order = {
        "revenue": 150.0,
        "freight": 45.0,
        "items_count": 1,
        "product_weight_g": 2000.0,
        "product_volume": 15000.0,
        "customer_state": "BA",
        "seller_state": "SP",
        "purchase_month": 11,
        "purchase_dayofweek": 4
    }
    
    predictor = DeliveryPredictor()
    print("Incoming Order Details:")
    print(json.dumps(dummy_order, indent=2))
    
    print("\nModel Prediction:")
    prediction = predictor.predict(dummy_order)
    print(json.dumps(prediction, indent=2))



Model loaded successfully.
Incoming Order Details:
{
  "revenue": 150.0,
  "freight": 45.0,
  "items_count": 1,
  "product_weight_g": 2000.0,
  "product_volume": 15000.0,
  "customer_state": "BA",
  "seller_state": "SP",
  "purchase_month": 11,
  "purchase_dayofweek": 4
}

Model Prediction:
{
  "is_late_prediction": 1,
  "probability_late": 0.6051450971769047,
  "risk_level": "High",
  "recommended_action": "Subsidize Expedited Shipping"
}


### Interactive UI
Run the cell below to launch the interactive predictor. Ensure you have run the cell above first to define the `DeliveryPredictor` class.

In [2]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Create the predictor instance
predictor = DeliveryPredictor()

# UI Components (Sliders and Dropdowns)
revenue = widgets.FloatSlider(value=150.0, min=10.0, max=5000.0, step=10.0, description='Revenue (R$):')
freight = widgets.FloatSlider(value=45.0, min=0.0, max=500.0, step=5.0, description='Freight (R$):')
items_count = widgets.IntSlider(value=1, min=1, max=20, step=1, description='Items Count:')
product_weight_g = widgets.FloatSlider(value=2000.0, min=50.0, max=30000.0, step=100.0, description='Weight (g):')
product_volume = widgets.FloatSlider(value=15000.0, min=100.0, max=300000.0, step=1000.0, description='Volume (cm³):')

states = ['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']
customer_state = widgets.Dropdown(options=states, value='BA', description='Cust. State:')
seller_state = widgets.Dropdown(options=states, value='SP', description='Sell. State:')
purchase_month = widgets.Dropdown(options=list(range(1, 13)), value=11, description='Month (1-12):')
purchase_dayofweek = widgets.Dropdown(options=list(range(0, 7)), value=4, description='Day (0-6):')

button = widgets.Button(description='Predict Delivery Risk', button_style='primary')
output = widgets.Output()

def on_button_clicked(b):
    with output:
        output.clear_output()
        order_data = {
            "revenue": revenue.value,
            "freight": freight.value,
            "items_count": items_count.value,
            "product_weight_g": product_weight_g.value,
            "product_volume": product_volume.value,
            "customer_state": customer_state.value,
            "seller_state": seller_state.value,
            "purchase_month": purchase_month.value,
            "purchase_dayofweek": purchase_dayofweek.value
        }
        
        prediction = predictor.predict(order_data)
        
        color = "red" if prediction['is_late_prediction'] == 1 else "green"
        status = "LATE" if prediction['is_late_prediction'] == 1 else "ON TIME"
        
        html_str = f"""
        <div style='border: 2px solid {color}; padding: 15px; border-radius: 10px; background-color: #f8f9fa; max-width: 400px;'>
            <h3 style='margin-top:0; color: {color};'>Prediction: {status}</h3>
            <b>Probability of Delay:</b> {prediction['probability_late']:.1%}<br>
            <b>Risk Level:</b> {prediction['risk_level']}<br>
            <b>Recommended Action:</b> <span style='background-color: yellow; padding: 2px 5px;'>{prediction['recommended_action']}</span>
        </div>
        """
        display(HTML(html_str))

button.on_click(on_button_clicked)

# Layout
ui_title = widgets.HTML("<h2>Interactive Delivery Risk Predictor</h2><p>Adjust the parameters below to see how the model reacts.</p>")
left_box = widgets.VBox([revenue, freight, product_weight_g, product_volume, items_count])
right_box = widgets.VBox([customer_state, seller_state, purchase_month, purchase_dayofweek])
ui = widgets.VBox([ui_title, widgets.HBox([left_box, right_box]), button, output])

display(ui)



Model loaded successfully.
